# Crypto ML Factor Cross-Section Backtest
**Strategy:** `crypto-ml-factor-cross-section`  
**Paper:** *Predicting Cryptocurrency Returns with Machine Learning*  
**Universe:** CoinGecko top-150 coins by market cap (dynamic, stablecoins excluded, ≥ $10M filter applied weekly)  
**Data sources:** CoinGecko (price/volume/mcap · cached) · Coin Metrics community (on-chain) · yfinance (macro) · Fama-French · pytrends  
**Factors:** up to 16/25 (TV + NTR now available for ALL coins; AA/NT/MRR/NF/TF from CM for major coins only)  
**Rebalance:** Weekly · **Portfolio:** Long-short decile 10 vs decile 1, value-weighted

In [ ]:
# Cell 1 — Setup + Data Load
# Stage 1 • skill: skills/01-data.md
# Deviations from spec (registered):
#   - Universe/prices/vol : CoinGecko snapshot (1 API call) → yfinance batch download
#   - MCAP history        : price × (current_mcap / current_price) — approx constant supply
#   - TV                  : yfinance USD volume for ALL coins (no per-coin API calls)
#   - NF/TF               : CM FlowInExNtv/FlowOutExNtv (top-50 major coins; 6s timeout)
#   - Missing factors     : NA, AAR, ZBA, LHN, LHI, LHO — not in community tier
#   - GT/GTCX             : pytrends monthly; NaN fallback on 429
#   - EPU                 : FRED USEPUINDXD (daily). TEU not found on FRED.
#   - PIT lag             : lag=1 day on all on-chain + macro features

# ── 0. INSTALLS ───────────────────────────────────────────────────────────────
import subprocess, sys
for pkg in ["coinmetrics-api-client", "pytrends", "pandas-datareader"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# ── 1. IMPORTS ────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
import requests
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader.data as pdr
from coinmetrics.api_client import CoinMetricsClient
from pytrends.request import TrendReq
from backtest.data import DataPanel, FieldSpec

# ── 2. PARAMETERS ─────────────────────────────────────────────────────────────
START         = "2014-01-01"
END           = "2026-05-20"
MIN_MCAP_USD  = 10_000_000
LAG_BARS      = 1
UNIVERSE_SIZE = 150     # top N non-stable coins by market cap
CM_MAX_ASSETS = 50      # CM community only covers major coins
CM_TIMEOUT    = 6       # seconds per CM API call

STABLECOINS = frozenset({
    "tether", "usd-coin", "binance-usd", "dai", "true-usd", "frax", "usdd",
    "tusd", "usdp", "gusd", "stasis-eurs", "neutrino", "paxos-standard",
    "liquity-usd", "fei-usd", "usdn", "celo-dollar", "nusd", "origin-dollar",
    "paypal-usd", "euro-coin", "angle-protocol", "terra-luna",
    "terrausd", "wrapped-steth", "staked-ether", "wrapped-bitcoin", "wrapped-ether",
})

# ── 3a. CoinGecko snapshot — ONE API call, no per-coin fetching ───────────────
def _cg_snapshot(n, stables=STABLECOINS):
    """Get top-n non-stable coins — current symbol, mcap, price. Just 1–2 API calls."""
    result, page = [], 1
    while len(result) < n + 100:
        try:
            resp = requests.get(
                "https://api.coingecko.com/api/v3/coins/markets",
                params={"vs_currency": "usd", "order": "market_cap_desc",
                        "per_page": 250, "page": page, "sparkline": False},
                timeout=30)
            if resp.status_code == 429:
                print("  CG 429 — sleeping 60s"); time.sleep(60); continue
            resp.raise_for_status()
            batch = resp.json()
        except Exception as e:
            print(f"  CG page {page} failed: {e}"); break
        result.extend(batch)
        if len(batch) < 250: break
        page += 1
        time.sleep(2.0)
    seen, out = set(), []
    for c in result:
        sym = c["symbol"].upper()
        if c["id"] not in stables and sym not in seen:
            seen.add(sym)
            out.append((sym, c.get("market_cap") or 0, c.get("current_price") or np.nan))
    return out[:n]

print(f"Fetching top-{UNIVERSE_SIZE} coins from CoinGecko (snapshot, 1-2 API calls)...")
universe_snapshot = _cg_snapshot(UNIVERSE_SIZE)   # [(SYM, cur_mcap, cur_price), ...]
raw_syms = [sym for sym, _, _ in universe_snapshot]
print(f"  Got {len(raw_syms)} coins: {raw_syms[:12]}...")

# ── 3b. yfinance batch download — all coins in ONE call, no rate limits ────────
yf_tickers = [f"{sym}-USD" for sym in raw_syms]
print(f"\nDownloading {len(yf_tickers)} crypto tickers from yfinance (batch)...")
raw = yf.download(yf_tickers, start=START, end=END, progress=False, auto_adjust=True)

prices_raw = raw["Close"].rename(columns=lambda c: c.replace("-USD", ""))
volume_raw = raw["Volume"].rename(columns=lambda c: c.replace("-USD", ""))
prices_raw.index = pd.to_datetime(prices_raw.index)
volume_raw.index = pd.to_datetime(volume_raw.index)

# Drop coins with >90% NaN prices (not available on Yahoo Finance)
nan_pct = prices_raw.isna().mean()
valid   = nan_pct[nan_pct < 0.90].index.tolist()
prices  = prices_raw[valid].copy()
volume  = volume_raw[valid].copy()
SYMS    = list(prices.columns)
print(f"  {len(SYMS)}/{len(raw_syms)} coins have data on Yahoo Finance")
print(f"  Skipped ({len(raw_syms)-len(SYMS)}): {[s for s in raw_syms if s not in SYMS][:10]}")
print(f"  Prices: {prices.shape}  NaN: {prices.isna().mean().mean():.1%}")

# ── 3c. Estimate historical market cap ────────────────────────────────────────
# mcap_t ≈ price_t × (current_mcap / current_price) — constant circulating supply approx.
# Good enough for $10M universe filter and value-weighting; price × ratio preserves
# cross-sectional mcap ordering at each point in time.
cur_info = {sym: (mc, cp) for sym, mc, cp in universe_snapshot}
mcap_d   = {}
for sym in SYMS:
    mc, cp = cur_info.get(sym, (np.nan, np.nan))
    if np.isfinite(cp) and cp > 0 and np.isfinite(mc) and mc > 0:
        mcap_d[sym] = prices[sym] * (mc / cp)
    else:
        mcap_d[sym] = pd.Series(np.nan, index=prices.index)
mcap = pd.DataFrame(mcap_d)
print(f"  MCAP coverage: {mcap.notna().mean().mean():.0%}  (estimated from price × supply ratio)")

# Use the yfinance index as the canonical date index
date_idx = prices.index

# ── 4. COIN METRICS — per-asset on-chain (AA, NT, MRR, NF, TF) ───────────────
# Only attempt top-CM_MAX_ASSETS coins; bounded by CM_TIMEOUT per call.
CM_METRICS = {
    "AdrActCnt":    "AA",
    "TxCnt":        "NT",
    "CapMVRVCur":   "MRR",
    "FlowInExNtv":  "FLIN",
    "FlowOutExNtv": "FLOUT",
}
cm = CoinMetricsClient(host="community-api.coinmetrics.io")

def _fetch_cm_metric(cm_client, syms, metric, start, end, date_idx,
                     sleep=0.15, call_timeout=CM_TIMEOUT, max_assets=CM_MAX_ASSETS):
    frames = []
    with ThreadPoolExecutor(max_workers=1) as exc:
        for sym in syms[:max_assets]:
            def _call(s=sym):
                df = cm_client.get_asset_metrics(
                    assets=[s.lower()], metrics=[metric],
                    start_time=start, end_time=end, frequency="1d",
                ).to_dataframe()
                df["time"] = pd.to_datetime(df["time"]).dt.tz_localize(None)
                return df.set_index("time")[[metric]].rename(columns={metric: s})
            try:
                df = exc.submit(_call).result(timeout=call_timeout)
                frames.append(df)
                time.sleep(sleep)
            except Exception:
                pass
    if not frames:
        return pd.DataFrame(index=date_idx, columns=prices.columns)
    return pd.concat(frames, axis=1).reindex(index=date_idx, columns=prices.columns)

print(f"\nFetching Coin Metrics (top-{CM_MAX_ASSETS} assets, {CM_TIMEOUT}s timeout/call)...")
cm_data = {}
for metric, label in CM_METRICS.items():
    df = _fetch_cm_metric(cm, SYMS, metric, START, END, date_idx)
    cov = df.notna().mean().mean()
    n   = df.notna().any().sum()
    print(f"  {label:6s}: {cov:.1%} coverage  {n} assets")
    cm_data[label] = df

flin  = cm_data.pop("FLIN",  None)
flout = cm_data.pop("FLOUT", None)

# ── 4b. DERIVED FEATURES ──────────────────────────────────────────────────────
# TV: yfinance volume — available for ALL coins (USD volume on Yahoo Finance crypto pairs)
cm_data["TV"] = volume.reindex(columns=prices.columns)
print(f"  TV    : {volume.notna().mean().mean():.1%} coverage (yfinance vol — all {len(SYMS)} coins)")

# NTR = mcap / TV  (all coins — mcap_approx / yfinance_vol; price roughly cancels)
tv_nz          = volume.reindex(columns=prices.columns).replace(0, np.nan)
ntr            = mcap.reindex(columns=prices.columns).div(tv_nz).replace([np.inf,-np.inf], np.nan)
cm_data["NTR"] = ntr
print(f"  NTR   : {ntr.notna().mean().mean():.1%} coverage (MCAP/TV — all coins)")

if flin is not None and flout is not None:
    cm_data["NF"] = flin.sub(flout)
    cm_data["TF"] = flin.add(flout)
    print(f"  NF    : {cm_data['NF'].notna().mean().mean():.1%} coverage (CM FlowIn-FlowOut)")
    print(f"  TF    : {cm_data['TF'].notna().mean().mean():.1%} coverage (CM FlowIn+FlowOut)")

# ── 5. MACRO FACTORS ──────────────────────────────────────────────────────────
print("\nFetching macro factors...")
MACRO_YF = {"^GSPC":"SPX","^VIX":"VIX","DX-Y.NYB":"USDX","GLD":"GLD","CL=F":"WTI"}
macro = {}
for ticker, label in MACRO_YF.items():
    s = yf.download(ticker, start=START, end=END, progress=False,
                    auto_adjust=True)["Close"].squeeze()
    s.index = pd.to_datetime(s.index)
    macro[label] = s.reindex(date_idx, method="ffill")
    print(f"  {label}: {macro[label].notna().mean():.0%}")

try:
    ff  = pdr.DataReader("F-F_Research_Data_Factors_daily", "famafrench", START, END)[0] / 100
    mom = pdr.DataReader("F-F_Momentum_Factor_daily",       "famafrench", START, END)[0] / 100
    ff.index  = pd.to_datetime(ff.index)
    mom.index = pd.to_datetime(mom.index)
    macro["RMRF"] = ff["Mkt-RF"].reindex(date_idx, method="ffill")
    macro["SMB"]  = ff["SMB"].reindex(date_idx, method="ffill")
    macro["UMD"]  = mom.iloc[:, 0].reindex(date_idx, method="ffill")
    print("  FF RMRF/SMB/UMD: OK")
except Exception as e:
    print(f"  FF FAIL: {e}")

print("\nFetching Google Trends...")
def _fetch_trends(keywords, label):
    try:
        pt = TrendReq(hl='en-US', tz=360, timeout=(10, 25))
        pt.build_payload(keywords, timeframe=f"{START} {END}")
        df = pt.interest_over_time()
        if df.empty: raise ValueError("empty")
        if 'isPartial' in df.columns: df = df.drop(columns=['isPartial'])
        s = df.mean(axis=1); s.index = pd.to_datetime(s.index)
        result = s.reindex(date_idx, method="ffill")
        print(f"  {label}: OK ({df.shape[0]} pts)")
        return result
    except Exception as e:
        print(f"  {label}: FAIL ({type(e).__name__}: {str(e)[:60]}) -- NaN")
        return pd.Series(np.nan, index=date_idx)

macro["GT"]   = _fetch_trends(["Bitcoin"], "GT")
time.sleep(30)
macro["GTCX"] = _fetch_trends(["Bitcoin","BTC","cryptocurrency","blockchain","crypto"], "GTCX")

print("\nFetching EPU from FRED...")
for label, sid in [("EPU","USEPUINDXD"), ("TEU","EPUITU")]:
    try:
        s = pdr.DataReader(sid, "fred", START, END).squeeze()
        s.index = pd.to_datetime(s.index)
        macro[label] = s.reindex(date_idx, method="ffill")
        print(f"  {label}: OK ({macro[label].notna().mean():.0%})")
    except Exception as e:
        print(f"  {label}: FAIL ({type(e).__name__}) -- NaN")
        macro[label] = pd.Series(np.nan, index=date_idx)

# ── 6. BUILD FEATURE DICT ─────────────────────────────────────────────────────
def _broadcast(s, date_idx, cols):
    arr = np.tile(s.reindex(date_idx, method="ffill").values[:, None], (1, len(cols)))
    return pd.DataFrame(arr, index=date_idx, columns=cols)

features = {}
for label, df in cm_data.items():
    features[label] = df.reindex(index=date_idx, columns=prices.columns)
for label, s in macro.items():
    features[label] = _broadcast(s, date_idx, prices.columns)
features["MCAP"] = mcap.reindex(columns=prices.columns)

volume = volume.reindex(index=date_idx, columns=prices.columns)

# ── 7. UNIVERSE MASK ──────────────────────────────────────────────────────────
universe = (mcap.reindex(columns=prices.columns) >= MIN_MCAP_USD).fillna(False)

# ── 8. FIELD SPECS ────────────────────────────────────────────────────────────
specs = {
    **{k: FieldSpec(lag=LAG_BARS, missing="ffill", max_staleness=7)  for k in cm_data},
    **{k: FieldSpec(lag=LAG_BARS, missing="ffill", max_staleness=3)
       for k in ["SPX","VIX","USDX","GLD","WTI","RMRF","SMB","UMD"]},
    **{k: FieldSpec(lag=LAG_BARS, missing="ffill", max_staleness=35)
       for k in ["GT","GTCX","EPU","TEU"]},
    "MCAP":   FieldSpec(lag=0, missing="ffill", max_staleness=7),
    "volume": FieldSpec(lag=0, missing="ffill", max_staleness=3),
}

# ── 9. BUILD DataPanel ────────────────────────────────────────────────────────
panel = DataPanel(
    prices, volume=volume,
    features=features, universe=universe,
    specs=specs, check_outliers=False,
)

print(f"\n-- DataPanel --")
print(f"  Bars    : {len(panel.dates)}")
print(f"  Assets  : {len(panel.assets_all)}")
print(f"  Range   : {panel.dates[0].date()} to {panel.dates[-1].date()}")
avail = [f for f in panel.available_fields() if f not in ('prices','volume','MCAP')]
print(f"  Factors : {len(avail)} -- {avail}")
prices.tail(3)

In [ ]:
# Cell 2 — Strategy Class
# Stage 2 • skill: skills/02-strategy.md
# Deviations from spec (registered):
#   - OLS-3 factors  : MRR, AA, NT  (NT proxies for NA — new addresses paywalled)
#   - gross_exposure : 0.5  (50% long / 50% short, net = 0)
#   - Factors live   : up to 16/25 (dynamic — drops unavailable or >95%-NaN features)
#   - TV/NTR         : from CoinGecko for ALL 150 coins (real cross-sectional signal)
#   - AA/NT/MRR/NF/TF: from CM community for major coins only; NaN for rest → imputed
#   - NF/TF          : use weekly diff, not pct_change (can be negative)

import warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from backtest.strategy import Strategy
from backtest.data import DataView

FACTOR_COLS = [
    # On-chain per-asset (Coin Metrics community)
    "AA", "NT", "MRR",       # active addr, tx count, MVRV
    "TV", "NTR",              # CoinGecko vol + NVT ratio — available for ALL coins
    "NF", "TF",               # exchange net/total flow (CM major coins; NaN elsewhere)
    # Market / Fama-French
    "RMRF", "SMB", "UMD",
    # Macro (yfinance)
    "SPX", "VIX", "USDX", "GLD", "WTI",
    # Sentiment + uncertainty (NaN fallback if fetch fails)
    "GT", "GTCX", "EPU", "TEU",
]
LEVEL_DIFF_FACTORS = frozenset({"NF", "TF"})   # can be negative → diff(), not pct_change()
RETURN_FACTORS     = frozenset({"RMRF", "SMB", "UMD"})  # daily returns → weekly sum
OLS3_COLS          = ["MRR", "AA", "NT"]

NN_LAYERS = {
    "nn1": (32,), "nn2": (32, 16), "nn3": (32, 16, 8),
    "nn4": (32, 16, 8, 4), "nn5": (32, 16, 8, 4, 2),
}
ALL_MODEL_TYPES = [
    "ols", "ols3", "pls", "lasso", "ridge", "enet",
    "rf", "gbrt", "knn", "nn1", "nn2", "nn3", "nn4", "nn5",
]


def _make_model(model_type: str):
    mt = model_type.lower()
    if mt in ("ols", "ols3"): return LinearRegression()
    if mt == "pls":           return PLSRegression(n_components=3)
    if mt == "lasso":         return Lasso(alpha=1e-3, max_iter=5000)
    if mt == "ridge":         return Ridge(alpha=1.0)
    if mt == "enet":          return ElasticNet(alpha=1e-3, l1_ratio=0.5, max_iter=5000)
    if mt == "rf":
        return RandomForestRegressor(n_estimators=200, max_depth=6,
                                     min_samples_leaf=5, n_jobs=-1, random_state=42)
    if mt == "gbrt":
        return GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                         learning_rate=0.05, subsample=0.8, random_state=42)
    if mt == "knn":           return KNeighborsRegressor(n_neighbors=5, weights="distance", n_jobs=-1)
    if mt in NN_LAYERS:
        return MLPRegressor(hidden_layer_sizes=NN_LAYERS[mt], max_iter=500,
                            early_stopping=True, validation_fraction=0.1,
                            random_state=42, learning_rate_init=1e-3)
    raise ValueError(f"Unknown model_type: {model_type!r}. Choose from {ALL_MODEL_TYPES}")


class CryptoMLFactorStrategy(Strategy):
    """
    Long D10 / short D1 value-weighted decile portfolio, rebalanced weekly (W-SUN).
    Replicates Gu et al. crypto ML factor paper with up to 16/25 free factors.

    Features are selected dynamically at fit() — any factor absent from the panel
    or >95% NaN in training is automatically dropped. generate_weights() uses
    exactly the features the model was trained on (_fitted_cols_).

    train_start: if set, fit() restricts to [train_start, train_end] (rolling window).
    """
    rebalance_frequency = "W-SUN"

    def __init__(self, model_type="rf", min_train_samples=100, n_deciles=10,
                 gross_exposure=0.5, train_start=None):
        if model_type.lower() not in ALL_MODEL_TYPES:
            raise ValueError(f"model_type must be one of {ALL_MODEL_TYPES}")
        self.model_type        = model_type.lower()
        self.min_train_samples = min_train_samples
        self.n_deciles         = n_deciles
        self.gross_exposure    = gross_exposure
        self.train_start       = pd.Timestamp(train_start) if train_start is not None else None
        self.feature_cols_     = OLS3_COLS if self.model_type == "ols3" else FACTOR_COLS
        self._pipeline         = None
        self._fitted_cols_     = []

    def __repr__(self):
        return (f"CryptoMLFactorStrategy(model_type={self.model_type!r}, "
                f"n_deciles={self.n_deciles}, gross_exposure={self.gross_exposure})")

    def required_data(self):
        return ["volume", "MCAP"]   # factors handled dynamically

    def _to_weekly(self, daily_map):
        out = {}
        for col, df in daily_map.items():
            if col in RETURN_FACTORS:
                out[col] = df.resample("W-SUN").sum()
            elif col in LEVEL_DIFF_FACTORS:
                # NF/TF can be negative — use week-over-week difference, not pct_change
                weekly = df.resample("W-SUN").last()
                out[col] = weekly.diff()
            else:
                w = df.resample("W-SUN").last().pct_change()
                out[col] = w.replace([np.inf, -np.inf], np.nan)
        return out

    def _build_Xy(self, weekly, prices, feature_cols):
        fwdret  = prices.resample("W-SUN").last().pct_change().shift(-1)
        fwdret  = fwdret.replace([np.inf, -np.inf], np.nan)
        ref_idx = weekly[feature_cols[0]].index
        weeks   = sorted(set(fwdret.index[:-1]) & set(ref_idx))
        Xs, ys  = [], []
        for dt in weeks:
            y_row = fwdret.loc[dt].dropna()
            if len(y_row) < 3:
                continue
            assets = y_row.index
            X_cs = pd.DataFrame(
                {col: weekly[col].loc[dt] if dt in weekly[col].index
                       else pd.Series(np.nan, index=prices.columns)
                 for col in feature_cols}
            ).loc[assets]
            valid = assets[X_cs.notna().any(axis=1) & y_row.notna()]
            if len(valid) < 3:
                continue
            Xs.append(X_cs.loc[valid].to_numpy(dtype=float, na_value=np.nan))
            ys.append(y_row.loc[valid].to_numpy(dtype=float, na_value=np.nan))
        if not Xs:
            return np.empty((0, len(feature_cols))), np.empty(0)
        return np.vstack(Xs), np.concatenate(ys)

    def fit(self, data):
        avail = [col for col in self.feature_cols_ if data.has_feature(col)]
        if not avail:
            self._pipeline = None; return

        daily  = {col: data.feature(col) for col in avail}
        prices = data.prices
        if self.train_start is not None:
            daily  = {col: df.loc[self.train_start:] for col, df in daily.items()}
            prices = prices.loc[self.train_start:]

        weekly  = self._to_weekly(daily)
        n_weeks = weekly[avail[0]].dropna(how="all").shape[0]
        if n_weeks < self.min_train_samples:
            self._pipeline = None; return

        X, y = self._build_Xy(weekly, prices, avail)
        if len(X) == 0:
            self._pipeline = None; return

        # Drop rows where y is still inf/nan (safety guard after _build_Xy filters)
        finite_mask = np.isfinite(y)
        X, y = X[finite_mask], y[finite_mask]
        if len(X) == 0:
            self._pipeline = None; return

        nan_pct = np.isnan(X).mean(axis=0)
        keep    = np.where(nan_pct < 0.95)[0]
        if len(keep) == 0:
            self._pipeline = None; return
        X = X[:, keep]
        self._fitted_cols_ = [avail[i] for i in keep]

        base = _make_model(self.model_type)
        if isinstance(base, PLSRegression):
            nc   = min(3, len(self._fitted_cols_), X.shape[1])
            base = PLSRegression(n_components=nc)
        self._pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
            ("model",   base),
        ])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            self._pipeline.fit(X, y)

    def generate_weights(self, data: DataView, t: pd.Timestamp) -> pd.Series:
        if self._pipeline is None or not self._fitted_cols_:
            return pd.Series(dtype=float)
        daily  = {col: data.feature(col) for col in self._fitted_cols_}
        weekly = self._to_weekly(daily)
        snap = {
            col: weekly[col].iloc[-1] if not weekly[col].empty
                 else pd.Series(np.nan, index=data.prices.columns)
            for col in self._fitted_cols_
        }
        X_now = pd.DataFrame(snap)
        valid  = X_now.index[X_now.notna().any(axis=1)]
        if len(valid) < self.n_deciles:
            return pd.Series(dtype=float)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            preds = self._pipeline.predict(
                X_now.loc[valid].to_numpy(dtype=float, na_value=np.nan)
            ).ravel()
        pred_s = pd.Series(preds, index=valid)
        ranks  = pred_s.rank(pct=True)
        lo     = 1.0 / self.n_deciles
        hi     = (self.n_deciles - 1) / self.n_deciles
        long_  = ranks[ranks >= hi].index
        short_ = ranks[ranks <= lo].index
        mcap   = data.feature("MCAP").iloc[-1]
        def _vw(idx):
            cap = mcap.loc[idx].clip(lower=0).fillna(0)
            tot = cap.sum()
            return cap / tot if tot > 0 else pd.Series(1.0 / len(idx), index=idx)
        w = pd.Series(0.0, index=valid)
        if len(long_):  w[long_]  =  _vw(long_)  * self.gross_exposure
        if len(short_): w[short_] = -_vw(short_) * self.gross_exposure
        return w

    def apply_risk(self, proposed, state, data):
        return proposed


# ── Validation ────────────────────────────────────────────────────────────────
strat_rf = CryptoMLFactorStrategy(model_type="rf")
print(repr(strat_rf))
print(f"rebalance      : {strat_rf.rebalance_frequency}")
print(f"feature_cols   : {strat_rf.feature_cols_}  ({len(strat_rf.feature_cols_)} declared)")
print(f"LEVEL_DIFF     : {sorted(LEVEL_DIFF_FACTORS)}")
print()
strat_ols3 = CryptoMLFactorStrategy(model_type="ols3")
print(f"OLS-3 features : {strat_ols3.feature_cols_}")
print()
print("All 14 model types instantiate OK:")
for mt in ALL_MODEL_TYPES:
    _ = CryptoMLFactorStrategy(model_type=mt)
    print(f"  {mt:6s} -> {type(_make_model(mt)).__name__}")

In [ ]:
# Cell 3 — Costs + Liquidity Cap
# Stage 3 • skill: skills/04-costs.md
# Model: realistic crypto execution costs (paper ignores costs — noted deviation)
#   Commission   : 10 bps  (0.10% taker fee, major exchange)
#   Spread       : 15 bps half-spread  (30 bps full bid-ask, large/mid-cap crypto)
#   MarketImpact : sqrt, k=10, 20-day ADV  (Almgren-style; weekly rebal keeps ratio low)
#   ShortBorrow  : 300 bps annual  (~3% p.a. on short notional, conservative crypto borrow)
#   LiquidityCap : 5% of 20-day ADV  (throttle outsized trades in thin assets)

from backtest.costs import (
    Commission, Spread, MarketImpact, ShortBorrow,
    CompositeCostModel, LiquidityCap,
)

costs = CompositeCostModel([
    Commission(bps=10),
    Spread(half_bps=15),
    MarketImpact(k=10, kind="sqrt", adv_lookback=20),
    ShortBorrow(annual_bps=300, trading_days=252),
])

liq = LiquidityCap(cap_pct=0.05, adv_lookback=20)

# ── Pre-flight checks ─────────────────────────────────────────────────────────
print("Cost components :", [type(m).__name__ for m in costs.models])
print(f"ShortBorrow rate : {300/252:.2f} bps/day  ({300} bps/yr)")
print(f"LiquidityCap     : {liq.cap_pct*100:.0f}% of {liq.adv_lookback}-day ADV")
print(f"Panel has volume : {panel.has_field('volume')}")   # must be True for Impact + Cap

# ── Estimated one-way transaction cost (before impact) ────────────────────────
one_way_bps = 10 + 15          # commission + half-spread
print(f"\nEstimated one-way cost (excl. impact): {one_way_bps} bps = {one_way_bps/100:.2f}%")
print(f"Estimated round-trip (excl. impact)  : {2*one_way_bps} bps = {2*one_way_bps/100:.2f}%")

In [ ]:
# Cell 4 — Rolling Walk-Forward Engine Run
# Stage 4 • skill: skills/05-engine-single-path.md
# Paper windows: train=4yr + val=3yr → 7yr rolling fit window (no HP tuning needed)
# OOS: 1yr per step, 1yr step → 6 non-overlapping windows (2021–2026)
# Each step: fresh strategy instance, retrained on trailing 7yr of data only.

import sys
sys.path.insert(0, r'C:\Users\User\backtest_engine')

from backtest.engine import Engine
import numpy as np
import pandas as pd

TRAIN_YRS  = 7   # 4yr paper train + 3yr paper validation
FIRST_TEST = 2021
LAST_TEST  = 2026

# ── Build windows ─────────────────────────────────────────────────────────────
windows = []
panel_end = panel.dates[-1]
for year in range(FIRST_TEST, LAST_TEST + 1):
    test_start = pd.Timestamp(f"{year}-01-01")
    test_end   = min(pd.Timestamp(f"{year}-12-31"), panel_end)
    fit_start  = test_start - pd.DateOffset(years=TRAIN_YRS)
    fit_end    = test_start - pd.Timedelta(days=1)
    train_dates = panel.dates[(panel.dates >= fit_start) & (panel.dates <= fit_end)]
    test_dates  = panel.dates[(panel.dates >= test_start) & (panel.dates <= test_end)]
    if len(train_dates) == 0 or len(test_dates) == 0:
        continue
    windows.append(dict(year=year, fit_start=fit_start, fit_end=fit_end,
                        test_start=test_start, test_end=test_end,
                        train_dates=train_dates, test_dates=test_dates))

print(f"Rolling walk-forward: {TRAIN_YRS}yr fit window, 1yr OOS step")
print(f"{'Year':>5}  {'Fit window':^25}  {'OOS window':^25}  {'Train':>6}  {'Test':>5}")
for w in windows:
    print(f"  {w['year']}  {str(w['fit_start'].date()):>12} -> {str(w['fit_end'].date()):<12}  "
          f"{str(w['test_start'].date()):>12} -> {str(w['test_end'].date()):<12}  "
          f"{len(w['train_dates']):>5}  {len(w['test_dates']):>5}")

# ── Run each window ───────────────────────────────────────────────────────────
results = []
strats  = []
for w in windows:
    print(f"\n[{w['year']}] Fit {w['fit_start'].date()} → {w['fit_end'].date()} "
          f"| OOS {w['test_start'].date()} → {w['test_end'].date()}")
    strat  = CryptoMLFactorStrategy(model_type="rf", train_start=w["fit_start"])
    engine = Engine(panel, strat, costs=costs, liquidity=liq, initial_capital=1_000_000)
    result = engine.run(train_dates=w["train_dates"], test_dates=w["test_dates"])
    tr     = result.equity_curve.iloc[-1] / result.metadata["initial_capital"] - 1
    fitted = strat._fitted_cols_
    print(f"  Return: {tr:>+8.2%} | Rebals: {result.metadata['n_rebalances']} | "
          f"Costs: ${result.costs.sum():>9,.0f} | Factors: {len(fitted)}")
    print(f"  Fitted: {fitted}")
    results.append((w["year"], result))
    strats.append(strat)

# ── Stitch OOS returns into one continuous equity curve ───────────────────────
all_rets        = pd.concat([r.returns.dropna() for _, r in results]).sort_index()
stitched_equity = (1 + all_rets).cumprod() * 1_000_000
total_ret       = stitched_equity.iloc[-1] / 1_000_000 - 1
span_days       = (all_rets.index[-1] - all_rets.index[0]).days
ann_ret         = (1 + total_ret) ** (365 / span_days) - 1
ann_vol         = all_rets.std() * np.sqrt(252)
sharpe          = ann_ret / ann_vol if ann_vol > 0 else np.nan
peak            = np.maximum.accumulate(stitched_equity.values)
max_dd          = ((stitched_equity.values - peak) / np.where(peak > 0, peak, 1)).min()
total_costs     = sum(r.costs.sum() for _, r in results)

print(f"\n{'='*60}")
print(f"Rolling Walk-Forward  (RF · {TRAIN_YRS}yr fit · 1yr step · {len(windows)} windows)")
print(f"Universe  : {len(SYMS)} coins  (CoinGecko top-{UNIVERSE_SIZE}, dynamic)")
print(f"OOS period: {all_rets.index[0].date()} to {all_rets.index[-1].date()}")
print(f"{'='*60}")
print(f"Total return     : {total_ret:>+10.2%}")
print(f"Ann. return      : {ann_ret:>+10.2%}")
print(f"Ann. volatility  : {ann_vol:>+10.2%}")
print(f"Sharpe ratio     : {sharpe:>+10.2f}")
print(f"Max drawdown     : {max_dd:>+10.2%}")
print(f"Final equity     : ${stitched_equity.iloc[-1]:>12,.0f}")
print(f"Total costs      : ${total_costs:>12,.0f}")
print(f"\n{'Year':>5}  {'Return':>8}  {'Vol':>7}  {'SR':>6}  {'MaxDD':>8}  {'Costs':>10}  {'Rebals':>6}  {'Factors':>8}")
for (year, res), strat in zip(results, strats):
    r      = res.returns.dropna()
    yr_ret = res.equity_curve.iloc[-1] / res.metadata["initial_capital"] - 1
    yr_vol = r.std() * np.sqrt(252)
    yr_sr  = (r.mean() * 252) / yr_vol if yr_vol > 0 else np.nan
    pk     = np.maximum.accumulate(res.equity_curve.values)
    yr_dd  = ((res.equity_curve.values - pk) / np.where(pk > 0, pk, 1)).min()
    print(f"  {year}  {yr_ret:>+8.2%}  {yr_vol:>7.2%}  {yr_sr:>6.2f}  "
          f"{yr_dd:>+8.2%}  ${res.costs.sum():>9,.0f}  {res.metadata['n_rebalances']:>6}  "
          f"{len(strat._fitted_cols_):>7}")

In [ ]:
# Cell 5 — Walk-Forward via CPCV + MultiPathEngine
# Stage 6 • skills/06-splitters.md + skills/07-multipath.md
# Config: 6 groups × ~710 bars, 2 test groups → C(6,2)=15 splits, C(5,1)=5 paths
#         purge=7 bars (1 week), embargo≈21 bars (~1 month)
# Factory pattern ensures each split gets a fresh, unfitted RF instance.

from backtest.splitters import CombinatorialPurgedCV
from backtest.engine import MultiPathEngine

# ── Splitter ──────────────────────────────────────────────────────────────────
splitter = CombinatorialPurgedCV(
    n_splits=6,
    n_test_groups=2,
    purge_bars=7,
    embargo_pct=0.005,
)

n            = len(panel.dates)
embargo_bars = int(n * splitter.embargo_pct)
print(f"Timeline : {n} bars  ({panel.dates[0].date()} to {panel.dates[-1].date()})")
print(f"Groups   : {splitter.n_groups} x ~{n // splitter.n_groups} bars each (~{n // splitter.n_groups / 365:.1f} yrs)")
print(f"Splits   : {splitter.n_splits()}  (trainings / engine runs)")
print(f"Paths    : {splitter.n_paths()}  (independent equity curves)")
print(f"Purge    : {splitter.purge_bars} bars | Embargo: {embargo_bars} bars | Trailing buffer: {splitter.purge_bars + embargo_bars} bars")

first_train, first_test = next(iter(splitter.split(panel.dates)))
print(f"\nFirst split:")
print(f"  train : {len(first_train)} bars  ({first_train[0].date()} to {first_train[-1].date()})")
print(f"  test  : {len(first_test)} bars  ({first_test[0].date()} to {first_test[-1].date()})")
print(f"  disjoint: {len(first_train.intersection(first_test)) == 0}")

# ── MultiPathEngine ───────────────────────────────────────────────────────────
print("\nRunning MultiPathEngine (15 splits x RF)...")
mp = MultiPathEngine(
    panel,
    lambda: CryptoMLFactorStrategy(model_type="rf"),
    splitter,
    costs=costs,
    liquidity=liq,
    initial_capital=1_000_000,
)
res = mp.run(n_jobs=1)   # n_jobs=1 surfaces errors inline before going parallel

print(f"\nSplits run  : {res.n_splits}")
print(f"Paths       : {res.n_paths}")
print(f"path_returns: {res.path_returns.shape}")
print(f"Date range  : {res.path_returns.index[0].date()} to {res.path_returns.index[-1].date()}")
print(f"NaN per path (expected ~{splitter.n_groups}): {res.path_returns.isna().sum().to_dict()}")
print()
print("Final equity per path:")
for col, val in res.path_equity.iloc[-1].items():
    tr = val / 1_000_000 - 1
    print(f"  {col}: ${val:>12,.0f}  ({tr:+.1%})")